# Step 1.3.1 — Populate and Validate RAND HRS Agent Metadata

This notebook loads **Codebook-verified** metadata into the six Delta tables created in Step 1.3. It uses `variable_id` generated by `hrs_variable_catalog` and looks up that ID for child tables. Do not load unreviewed regex matches directly: Codebook tables and headings can create false positives.

First set `CODEBOOK_VERSION`, then populate the approved lists with exact values and source pages verified against the Codebook. The notebook uses Delta `MERGE` for rerunnable loads and validates logical keys afterward.

In [ ]:
from pyspark.sql import functions as F, types as T

CATALOG = 'dev_catalog'
SCHEMA = 'hrs_agent_metadata'
CODEBOOK_VERSION = '1992-2022v1'
VARIABLES = f'{CATALOG}.{SCHEMA}.hrs_variable_catalog'
WAVES = f'{CATALOG}.{SCHEMA}.hrs_variable_wave_coverage'
COHORTS = f'{CATALOG}.{SCHEMA}.hrs_variable_cohort_coverage'
STATS = f'{CATALOG}.{SCHEMA}.hrs_variable_statistics'
LABELS = f'{CATALOG}.{SCHEMA}.hrs_variable_value_labels'
CHUNKS = f'{CATALOG}.{SCHEMA}.hrs_document_chunks'

assert CODEBOOK_VERSION != '1992-2022v1', 'Set the actual Codebook version.'
for table in [VARIABLES, WAVES, COHORTS, STATS, LABELS, CHUNKS]:
    if not spark.catalog.tableExists(table):
        raise ValueError(f'Required table does not exist: {table}')


## 1. Discover candidate variable mentions for review

This is a broad candidate finder, not a definitive parser. Review results before loading anything into the final metadata tables.

In [ ]:
import re
PAGES = f'{CATALOG}.{SCHEMA}.hrs_document_pages'
CANDIDATES = f'{CATALOG}.{SCHEMA}.hrs_variable_candidates'
pages_df = spark.table(PAGES)
required_cols = {'page_number', 'page_text'}
if not required_cols.issubset(set(pages_df.columns)):
    raise ValueError(f'{PAGES} must contain columns {sorted(required_cols)}; update the column mapping.')
if 'codebook_version' in pages_df.columns:
    pages_df = pages_df.filter(F.col('codebook_version') == CODEBOOK_VERSION)
token_re = re.compile(r'(?<![A-Za-z0-9_])([A-Z][A-Z0-9_]{1,19})(?![A-Za-z0-9_])')
ignore = {'THE', 'AND', 'FOR', 'FROM', 'WITH', 'WAVE', 'PAGE', 'TABLE', 'NOTE'}
candidate_rows = []
cols = ['page_number', 'page_text'] + (['document_id'] if 'document_id' in pages_df.columns else [])
for r in pages_df.select(*cols).filter(F.col('page_text').isNotNull()).toLocalIterator():
    text = r['page_text'] or ''
    seen = set()
    for m in token_re.finditer(text):
        token = m.group(1)
        if token in ignore or token in seen:
            continue
        seen.add(token)
        candidate_rows.append((str(r['document_id']) if 'document_id' in r.asDict() and r['document_id'] else 'rand_hrs_codebook', int(r['page_number']), token, text[max(0,m.start()-160):min(len(text),m.end()+220)], CODEBOOK_VERSION))
candidate_schema = T.StructType([T.StructField('document_id',T.StringType(),False), T.StructField('source_page',T.IntegerType(),False), T.StructField('candidate_token',T.StringType(),False), T.StructField('context_excerpt',T.StringType(),True), T.StructField('codebook_version',T.StringType(),False)])
candidate_df = spark.createDataFrame(candidate_rows, candidate_schema)
candidate_df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(CANDIDATES)
print(f'Wrote {len(candidate_rows):,} candidate mentions to {CANDIDATES}')
display(candidate_df.orderBy('candidate_token','source_page').limit(300))


## 1. Approved input records

Replace the empty lists with reviewed Codebook values. Use only verified source page numbers. Leave cohort coverage empty unless cohort-specific applicability is documented. Statistics are by wave only.

In [ ]:
approved_variables = [
    # {'variable_name': 'R1BMI', 'variable_label': 'verified label', 'definition': 'verified definition',
    #  'survey_section': 'Health', 'measurement_unit': 'kg/m² if documented', 'source_page': 123}
]
approved_wave_coverage = [
    # {'variable_name': 'R1BMI', 'wave_number': 1, 'survey_year': 1992,
    #  'coverage_status': 'Available', 'source_page': 123}
]
approved_cohort_coverage = [
    # {'variable_name': 'VARIABLE', 'hacohort_number': 1, 'coverage_status': 'Available', 'source_page': 123}
]
approved_statistics = [
    # {'variable_name': 'R1BMI', 'wave_number': 1, 'statistic_name': 'Mean',
    #  'statistic_value': 27.4, 'source_page': 123}  # Replace with actual verified value.
]
approved_value_labels = [
    # {'variable_name': 'VARIABLE', 'value_code': '1', 'value_label': 'verified label',
    #  'wave_number': None, 'source_page': 123}
]
approved_chunks = [
    # {'document_id': 'rand_hrs_codebook', 'page_number': 123, 'chunk_index': 1,
    #  'section_heading': 'verified heading', 'chunk_text': 'verified Codebook passage'}
]

print({'variables': len(approved_variables), 'wave_coverage': len(approved_wave_coverage),
       'cohort_coverage': len(approved_cohort_coverage), 'statistics': len(approved_statistics),
       'value_labels': len(approved_value_labels), 'chunks': len(approved_chunks)})


## 2. Upsert the variable catalog

The target table generates `variable_id`. This merge uses the logical key `variable_name + codebook_version`.

In [ ]:
variable_schema = T.StructType([
    T.StructField('variable_name', T.StringType(), False),
    T.StructField('variable_label', T.StringType(), True),
    T.StructField('definition', T.StringType(), True),
    T.StructField('survey_section', T.StringType(), True),
    T.StructField('measurement_unit', T.StringType(), True),
    T.StructField('source_page', T.IntegerType(), True),
    T.StructField('codebook_version', T.StringType(), False),
])
var_rows = [{**x, 'variable_name': x['variable_name'].strip(), 'codebook_version': CODEBOOK_VERSION} for x in approved_variables]
if len({(x['variable_name'], x['codebook_version']) for x in var_rows}) != len(var_rows):
    raise ValueError('Duplicate variable_name + codebook_version in approved_variables.')
if var_rows:
    var_df = spark.createDataFrame(var_rows, variable_schema)
    var_df.createOrReplaceTempView('approved_variables_src')
    spark.sql(f'''MERGE INTO {VARIABLES} t USING approved_variables_src s
      ON t.variable_name=s.variable_name AND t.codebook_version=s.codebook_version
      WHEN MATCHED THEN UPDATE SET t.variable_label=s.variable_label, t.definition=s.definition,
        t.survey_section=s.survey_section, t.measurement_unit=s.measurement_unit,
        t.source_page=s.source_page, t.updated_timestamp=current_timestamp()
      WHEN NOT MATCHED THEN INSERT (variable_name, variable_label, definition, survey_section,
        measurement_unit, source_page, codebook_version)
      VALUES (s.variable_name, s.variable_label, s.definition, s.survey_section,
        s.measurement_unit, s.source_page, s.codebook_version)''')

variable_lookup = (spark.table(VARIABLES).filter(F.col('codebook_version') == CODEBOOK_VERSION)
                   .select('variable_id', 'variable_name', 'codebook_version'))


## 3. Resolve variable IDs and merge related tables

This helper resolves `variable_id` from the catalog and fails if a child row references a variable that was not loaded.

In [ ]:
def resolve_rows(rows, fields):
    if not rows:
        return None
    schema = T.StructType([T.StructField('variable_name', T.StringType(), False)] +
        [T.StructField(name, dtype, True) for name, dtype in fields])
    src = spark.createDataFrame([{'variable_name': x['variable_name'].strip(), **{n: x.get(n) for n, _ in fields}} for x in rows], schema).alias('s')
    out = src.join(variable_lookup.alias('v'),
        (F.col('s.variable_name') == F.col('v.variable_name')) & (F.col('v.codebook_version') == CODEBOOK_VERSION), 'left')
    out = out.select(F.col('v.variable_id').alias('variable_id'), *[F.col('s.'+n).alias(n) for n, _ in fields])
    if out.filter(F.col('variable_id').isNull()).limit(1).count():
        raise ValueError('Child metadata references a variable not found in hrs_variable_catalog.')
    return out.withColumn('codebook_version', F.lit(CODEBOOK_VERSION))

def merge_rows(df, view, target, condition, update_set, insert_columns, insert_values):
    if df is None:
        return
    df.createOrReplaceTempView(view)
    spark.sql(f'''MERGE INTO {target} t USING {view} s ON {condition}
      WHEN MATCHED THEN UPDATE SET {update_set}
      WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})''')

wave_df = resolve_rows(approved_wave_coverage, [('wave_number', T.IntegerType()), ('survey_year', T.IntegerType()), ('coverage_status', T.StringType()), ('source_page', T.IntegerType())])
merge_rows(wave_df, 'wave_src', WAVES,
  't.variable_id=s.variable_id AND t.wave_number=s.wave_number AND t.codebook_version=s.codebook_version',
  't.survey_year=s.survey_year, t.coverage_status=s.coverage_status, t.source_page=s.source_page',
  'variable_id, wave_number, survey_year, coverage_status, source_page, codebook_version',
  's.variable_id, s.wave_number, s.survey_year, s.coverage_status, s.source_page, s.codebook_version')

cohort_df = resolve_rows(approved_cohort_coverage, [('hacohort_number', T.IntegerType()), ('coverage_status', T.StringType()), ('source_page', T.IntegerType())])
merge_rows(cohort_df, 'cohort_src', COHORTS,
  't.variable_id=s.variable_id AND t.hacohort_number=s.hacohort_number AND t.codebook_version=s.codebook_version',
  't.coverage_status=s.coverage_status, t.source_page=s.source_page',
  'variable_id, hacohort_number, coverage_status, source_page, codebook_version',
  's.variable_id, s.hacohort_number, s.coverage_status, s.source_page, s.codebook_version')

stats_df = resolve_rows(approved_statistics, [('wave_number', T.IntegerType()), ('statistic_name', T.StringType()), ('statistic_value', T.DoubleType()), ('source_page', T.IntegerType())])
merge_rows(stats_df, 'stats_src', STATS,
  't.variable_id=s.variable_id AND t.wave_number=s.wave_number AND t.statistic_name=s.statistic_name AND t.codebook_version=s.codebook_version',
  't.statistic_value=s.statistic_value, t.source_page=s.source_page',
  'variable_id, wave_number, statistic_name, statistic_value, source_page, codebook_version',
  's.variable_id, s.wave_number, s.statistic_name, s.statistic_value, s.source_page, s.codebook_version')

labels_df = resolve_rows(approved_value_labels, [('value_code', T.StringType()), ('value_label', T.StringType()), ('wave_number', T.IntegerType()), ('source_page', T.IntegerType())])
merge_rows(labels_df, 'labels_src', LABELS,
  't.variable_id=s.variable_id AND t.value_code=s.value_code AND coalesce(t.wave_number,-1)=coalesce(s.wave_number,-1) AND t.codebook_version=s.codebook_version',
  't.value_label=s.value_label, t.source_page=s.source_page',
  'variable_id, value_code, value_label, wave_number, source_page, codebook_version',
  's.variable_id, s.value_code, s.value_label, s.wave_number, s.source_page, s.codebook_version')
print('Variable-related metadata merges complete.')


## 4. Merge document chunks

Chunks are identified for reruns by document, page, chunk index, and Codebook version.

In [ ]:
chunk_schema = T.StructType([
    T.StructField('document_id', T.StringType(), False), T.StructField('page_number', T.IntegerType(), False),
    T.StructField('chunk_index', T.IntegerType(), True), T.StructField('section_heading', T.StringType(), True),
    T.StructField('chunk_text', T.StringType(), False), T.StructField('codebook_version', T.StringType(), False)])
chunk_rows = [{'document_id': x['document_id'], 'page_number': int(x['page_number']),
    'chunk_index': int(x.get('chunk_index', 1)), 'section_heading': x.get('section_heading'),
    'chunk_text': x['chunk_text'], 'codebook_version': CODEBOOK_VERSION} for x in approved_chunks]
if chunk_rows:
    chunks_df = spark.createDataFrame(chunk_rows, chunk_schema)
    chunks_df.createOrReplaceTempView('chunks_src')
    spark.sql(f'''MERGE INTO {CHUNKS} t USING chunks_src s
      ON t.document_id=s.document_id AND t.page_number=s.page_number
       AND coalesce(t.chunk_index,-1)=coalesce(s.chunk_index,-1) AND t.codebook_version=s.codebook_version
      WHEN MATCHED THEN UPDATE SET t.section_heading=s.section_heading, t.chunk_text=s.chunk_text
      WHEN NOT MATCHED THEN INSERT (document_id, page_number, chunk_index, section_heading, chunk_text, codebook_version)
      VALUES (s.document_id, s.page_number, s.chunk_index, s.section_heading, s.chunk_text, s.codebook_version)''')
print('Document chunk merge complete.')


## 5. Validate duplicate logical keys and orphan variable IDs

Each duplicate query should return zero rows. These are validation rules because Delta does not enforce the intended composite uniqueness.

In [ ]:
checks = {
 'variable_catalog': f'SELECT variable_name, codebook_version, COUNT(*) n FROM {VARIABLES} GROUP BY variable_name, codebook_version HAVING COUNT(*) > 1',
 'wave_coverage': f'SELECT variable_id, wave_number, codebook_version, COUNT(*) n FROM {WAVES} GROUP BY variable_id, wave_number, codebook_version HAVING COUNT(*) > 1',
 'cohort_coverage': f'SELECT variable_id, hacohort_number, codebook_version, COUNT(*) n FROM {COHORTS} GROUP BY variable_id, hacohort_number, codebook_version HAVING COUNT(*) > 1',
 'statistics': f'SELECT variable_id, wave_number, statistic_name, codebook_version, COUNT(*) n FROM {STATS} GROUP BY variable_id, wave_number, statistic_name, codebook_version HAVING COUNT(*) > 1',
 'value_labels': f'SELECT variable_id, value_code, coalesce(wave_number,-1) wave_key, codebook_version, COUNT(*) n FROM {LABELS} GROUP BY variable_id, value_code, coalesce(wave_number,-1), codebook_version HAVING COUNT(*) > 1',
 'chunks': f'SELECT document_id, page_number, coalesce(chunk_index,-1) chunk_key, codebook_version, COUNT(*) n FROM {CHUNKS} GROUP BY document_id, page_number, coalesce(chunk_index,-1), codebook_version HAVING COUNT(*) > 1'
}
for name, sql in checks.items():
    result = spark.sql(sql)
    n = result.count()
    print(f'{name}: duplicate groups = {n}')
    if n: display(result.limit(100))

known_ids = spark.table(VARIABLES).select('variable_id').distinct()
for name, table in [('wave_coverage', WAVES), ('cohort_coverage', COHORTS), ('statistics', STATS), ('value_labels', LABELS)]:
    n = spark.table(table).select('variable_id').join(known_ids, 'variable_id', 'left_anti').count()
    print(f'{name}: orphan variable_id rows = {n}')

for table in [VARIABLES, WAVES, COHORTS, STATS, LABELS, CHUNKS]:
    print(f'{table}: {spark.table(table).count():,} rows')


## Before Step 1.4

- Verify every loaded definition, statistic, value label, and coverage record against the original Codebook.
- Confirm duplicate-key groups and orphan variable IDs are zero.
- Confirm source page references are accurate.
- Do not infer cohort coverage where it is not documented.

This notebook provides controlled DML, not a fully automatic Codebook parser. The next refinement should use representative page text from the actual Codebook to improve automated extraction.